# Lab 04 — 行為序列建模與群體識別

**課程**：player-behavior-analytics（玩家行為分析）／第 4 課 行為序列建模與群體識別
**目的**：把玩家的「行為序列」壓縮成特徵向量，用 KMeans 讓演算法自動分群，再與真實原型對照——看機器發現的群體與設計原型有何異同。
**方式**：全程以 Gemini（AI 助手）產生程式碼——把每個任務的自然語言描述貼給 Gemini，再把生成的程式碼貼進下方 code cell 執行；**重點是觀察結果**，不是寫程式。每個任務下方附參考程式碼，可先自行生成再比對。

> 執行：在 Google Colab 上傳本 .ipynb（或直接開啟），Runtime → Run all。本範本內建模擬數據產生器，無需上傳檔案。

## 0. 載入數據

執行下方 cell 產生模擬數據。本 Lab 保留 archetype 欄位，作為最後一步的「真相」對照。

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    """模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    """
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df

df = gen_baccarat(seed=42)
df.head()


## 任務 1 — 序列特徵工程

金額變異（Lab 03）分不出「追注型」與「波動型」——因為它只看金額分布，不看順序。序列特徵把「順序結構」變成數字：

| 特徵 | 定義 | 捕捉的結構 |
|------|------|-----------|
| avg_bet / std_bet | 下注金額平均／標準差 | 金額規模與擺動 |
| change_rate | 每局金額平均變動幅度 ÷ 平均下注 | 金額變化的「節奏強度」 |
| switch_rate | 轉換下注類型的局數比例 | 是否反覆換邊 |
| post_loss_raise | 輸後加注的比例 | 輸後反應 |
| seq_entropy | 下注類型序列的資訊熵（÷log2(3) 正規化） | 下注方向的規律性 |

在 Gemini 輸入：

> 「對每位玩家計算：平均下注、下注標準差、金額平均變動率（|Δ金額| 的平均 ÷ 平均下注）、下注類型轉換率、輸後加注比例、下注類型序列的資訊熵（以 log2(3) 正規化）。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
d = df.sort_values(["player_id", "timestamp"]).copy()
d["prev_bt"] = d.groupby("player_id")["bet_type"].shift()
d["prev_win"] = d.groupby("player_id")["is_win"].shift()
d["avg_bet"] = d.groupby("player_id")["bet_amount"].transform("mean")

def seq_entropy(s):
    p = s.value_counts(normalize=True)
    return float(-(p * np.log2(p)).sum()) / np.log2(3)   # 0=全押同一邊, 1=完全均勻

feat = []
for pid, g in d.groupby("player_id"):
    amt = g["bet_amount"]
    feat.append({
        "player_id": pid,
        "avg_bet": amt.mean(),
        "std_bet": amt.std(),
        "change_rate": (amt.diff().abs() / g["avg_bet"].iloc[0]).mean(),
        "switch_rate": (g["bet_type"] != g["prev_bt"].fillna(g["bet_type"])).mean(),
        "post_loss_raise": g.loc[g["prev_win"] == False, "bet_amount"]
                            .gt(g["avg_bet"].iloc[0]).mean(),
        "seq_entropy": seq_entropy(g["bet_type"]),
    })
feat = pd.DataFrame(feat)
feat.round(3)

**觀察**：光看 std_bet 分不出追注型（P01）與波動型（P04）——兩者金額都大幅擺動；change_rate 拉開一些差距（本例 0.36 vs 0.60），但兩者在特徵空間仍可能落在相近位置。分群演算法如何取捨，看下一步。

## 任務 2 — 標準化 + KMeans 分群

在 Gemini 輸入：

> 「把上述特徵標準化（StandardScaler），用 KMeans 分成 3 群，把分群結果加回表格。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

X = feat.drop(columns=["player_id"])
Z = StandardScaler().fit_transform(X)
feat["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Z)
feat.round(3)

## 任務 3 — 解讀分群

在 Gemini 輸入：

> 「輸出每個分群的各特徵平均。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
profiles = feat.groupby("cluster")[
    ["avg_bet", "std_bet", "change_rate", "switch_rate", "post_loss_raise", "seq_entropy"]
].mean().round(3)
profiles

**觀察**：為每個分群命名（以表格數值為準，分群編號因種子而異）。本例（seed=42）的結果約為：
- cluster 0：固定金額、極低轉換率（switch ≈ 0.03）、低熵 → 「死守單邊的固定型」
- cluster 1：高金額變異、高 change_rate → 「高變動加注型」
- cluster 2：固定金額、隨機轉換 → 「紀律固定型」

## 任務 4 — 方法 vs 真相：與真實原型對照

在 Gemini 輸入：

> 「把分群結果與每位玩家的真實原型（archetype）交叉比對。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
truth = df[["player_id", "archetype"]].drop_duplicates()
feat.merge(truth, on="player_id").groupby(["cluster", "archetype"]).size().unstack(fill_value=0)

**討論**（本例 seed=42 的交叉表）：
- 演算法自動發現的 3 群 vs 設計時的 5 原型：哪些原型合併？哪些分開？
- 本例：謬誤型與紀律型金額完全相同（avg 500、std 0）——兩者靠 switch_rate／seq_entropy 分開（死守單邊 vs 隨機轉換）；追注型、順勢型與波動型合併成一群——單靠金額特徵不足以分離「規則性加注」與「隨機波動」
- 合併的含義：在可用特徵下，兩者行為「統計上不可分」——若營運上需要區分（例如謬誤型需要輔導、波動型需關注），就需要更多特徵（牌路節奏、側注等真實系統維度）
- 分裂的含義：同一原型內的行為多樣性比設計分類更細
- 對營運的意義：分群不需要預設標籤——從數據直接浮現的群體，就是個人化服務的起點

## 進階（選做）— 監督式驗證：以特徵預測原型

課程地圖提到 XGBoost——示範「特徵 → 監督學習」閉環。樣本僅 10 位玩家、5 原型，交叉驗證只能拆 2 折——此數字只示範管線，不具統計效力。

在 Gemini 輸入：

> 「用 XGBClassifier 對特徵做 2-fold 交叉驗證，預測玩家的原型。」

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt

ARCHETYPES = ["consistent", "chaser", "momentum", "fallacy", "volatile"]


def gen_baccarat(seed=42, n_players=10, days=3, sessions=2, rounds=30):
    '''模擬智慧娛樂桌百家樂下注紀錄（教學用生成器）。

    原型依序循環：consistent 紀律型 / chaser 追注型 / momentum 順勢型 /
    fallacy 謬誤型 / volatile 波動型。
    簡化：每位玩家的每局結果獨立抽籤（真實同桌玩家共享開牌結果）。
    '''
    rng = np.random.default_rng(seed)
    BT = ["Banker", "Player", "Tie"]
    P_OUT = [0.4586, 0.4462, 0.0952]   # 8 副牌百家樂開牌機率
    P_BET = [0.50, 0.45, 0.05]         # 下注類型偏好
    BASE = {"consistent": 500, "chaser": 500, "momentum": 500,
            "fallacy": 500, "volatile": 300}
    PATS = {"consistent": [(14, 0), (20, 0)],   # 各原型偏好時段
            "chaser": [(20, 0), (22, 0)],
            "momentum": [(14, 0), (16, 0)],
            "fallacy": [(18, 0), (20, 0)],
            "volatile": None}                   # None = 隨機時段
    first = dt.date(2026, 6, 1)

    def pay(bt, oc, amt):
        if bt == "Banker" and oc == "Banker":
            return round(amt * 0.95, 2)   # 莊贏抽 5% 佣金
        if bt == "Player" and oc == "Player":
            return amt
        if bt == "Tie" and oc == "Tie":
            return round(amt * 8.0, 2)    # 和局賠 8 倍
        return 0.0

    rows = []
    for p in range(n_players):
        arch = ARCHETYPES[p % len(ARCHETYPES)]
        cur, prev_win = BASE[arch], None
        streak_out, streak_len, last_bt = None, 0, None
        for d in range(days):
            day = first + dt.timedelta(days=d)
            for s in range(sessions):
                pat = PATS[arch]
                if pat:
                    hh, mm = pat[s % len(pat)]
                else:
                    hh, mm = int(rng.integers(12, 24)), int(rng.integers(0, 60))
                t0 = dt.datetime(day.year, day.month, day.day, hh, mm, 0)
                for r in range(rounds):
                    oc = rng.choice(BT, p=P_OUT)
                    # --- 下注類型 ---
                    if arch == "fallacy" and streak_len >= 3 and oc == streak_out:
                        # 相信「平衡定律」：連開 3 次同路後轉押另一邊
                        bt = (rng.choice(BT[:2]) if streak_out == "Tie"
                              else ("Player" if streak_out == "Banker" else "Banker"))
                    elif arch == "fallacy":
                        bt = last_bt if last_bt else rng.choice(BT, p=P_BET)
                    else:
                        bt = rng.choice(BT, p=P_BET)
                    # --- 下注金額 ---
                    if arch == "consistent" or arch == "fallacy":
                        cur = BASE[arch]
                    elif arch == "chaser":    # 輸後加注追趕，贏後回到 base
                        cur = BASE[arch] if prev_win is not False else min(BASE[arch] * 3, int(cur * 1.5))
                    elif arch == "momentum":  # 贏後順勢加注，輸後回到 base
                        cur = min(BASE[arch] * 3, int(cur * 1.4)) if prev_win else BASE[arch]
                    else:                     # volatile：忽大忽小
                        cur = int(rng.integers(1, 11)) * 100
                    rows.append({
                        "player_id": "P%02d" % p, "archetype": arch,
                        "session_id": "P%02d-D%d-S%d" % (p, d + 1, s + 1),
                        "table_id": "T%d" % (p % 2 + 1),
                        "timestamp": t0 + dt.timedelta(seconds=45 * r),
                        "round_no": r + 1, "bet_type": bt, "bet_amount": cur,
                        "outcome": oc, "payout": pay(bt, oc, cur)})
                    prev_win = pay(bt, oc, cur) > 0
                    streak_out, streak_len = (oc, streak_len + 1) if oc == streak_out else (oc, 1)
                    last_bt = bt
    df = pd.DataFrame(rows)
    df["is_win"] = df["payout"] > 0
    return df
try:
    from xgboost import XGBClassifier
    from sklearn.model_selection import cross_val_score

    y = (truth.set_index("player_id")
             .loc[feat["player_id"], "archetype"]
             .astype("category").cat.codes)   # XGBoost 需整數類別標籤
    acc = cross_val_score(XGBClassifier(n_estimators=50, random_state=42),
                          X, y, cv=2, n_jobs=1)
    print("2-fold 平均準確率: %.2f (±%.2f)" % (acc.mean(), acc.std()))
    print("注意：僅 10 位玩家，此數字只示範管線，不具統計效力。")
except ImportError:
    print("XGBoost 未安裝——在 Colab 執行 !pip install xgboost 後重跑本單元（選做）。")

## 小結 — 數據閉環

Lab 01–04 完整走過：原始紀錄 → 個體視角 → 指紋維度 → 變異性訊號 → 序列特徵 → 自動分群 → 對照驗證。第 5 課將把分群結果接到營運決策（流失預警、個人化優惠、VIP 服務）。